# Case: Eficiência Multicanal (TV e Digital)

Neste notebook, partilho a minha abordagem para a extração, tratamento e análise de dados das campanhas. O objetivo principal é cruzar os investimentos de TV Aberta e Digital para apurar métricas reais de performance, como o CAC e a Taxa de Conversão (CRO).

Como o meu foco é cruzar Dados e Produto (UI/UX), estruturei este código com foco na governança: a base tem de sair daqui totalmente limpa e à prova de erros matemáticos, pronta a ser consumida por um *dashboard* executivo.

## 1. Ingestão e Limpeza de Dados
O primeiro passo é carregar as bases e fazer uma higienização rápida. Como é comum existirem falhas de *tracking* que geram valores nulos, trato essas quebras diretamente com o Pandas (preenchendo com zero) para garantir a integridade dos cálculos.

Em seguida, subo os *dataframes* para uma base de dados SQLite em memória. Isto permite-nos usar SQL Avançado para as regras de negócio e correr os *joins* com alta performance.

In [1]:
import pandas as pd
import sqlite3

# Carregando os arquivos CSV para DataFrames
df_campaings = pd.read_csv('/content/drive/Othercomputers/Meu PC/Downloads/CASE_GLOBO/base_campaigns.csv')
df_digital =  pd.read_csv('/content/drive/Othercomputers/Meu PC/Downloads/CASE_GLOBO/base_digital.csv')
df_tv = pd.read_csv('/content/drive/Othercomputers/Meu PC/Downloads/CASE_GLOBO/base_tv.csv')

# Tratamento de nulos (preencher com 0 em colunas numéricas para garantir a matemática do SQL)
df_digital.fillna(0, inplace=True)
df_tv.fillna(0, inplace=True)

# Cria uma conexão com um banco de dados em memória
conn = sqlite3.connect(':memory:')

# Salva os DataFrames como tabelas SQL
df_campaings.to_sql('campaigns', conn, index=False, if_exists='replace')
df_digital.to_sql('digital', conn, index=False, if_exists='replace')
df_tv.to_sql('tv', conn, index=False, if_exists='replace')

3501

## 2. Modelação e Regras de Negócio (SQL)
Todo o trabalho pesado acontece nesta *query*.

* Utilizei **CTEs** para isolar e consolidar as métricas de TV e Digital antes de fazer o *join* com as campanhas, evitando linhas duplicadas.
* Em termos de governança, o uso do `COALESCE` é fundamental aqui: se uma campanha só rodou no digital, não podemos deixar que um valor nulo da TV quebre a soma total do investimento.
* Para as métricas de CRO, apliquei blocos de `CASE WHEN` para isolar erros de divisão por zero.

In [2]:
query = """
    WITH digital_metrics AS (
        -- Consolidando os dados de performance digital e aquisição
        SELECT
            campaign_id,
            SUM(impressions) AS total_impressions,
            SUM(clicks) AS total_clicks,
            SUM(conversions) AS total_conversions,
            SUM(digital_investment) AS total_digital_cost
        FROM digital
        GROUP BY campaign_id
    ),
    tv_metrics AS (
        -- Consolidando os investimentos e inserções offline
        SELECT
            campaign_id,
            SUM(audience) AS total_audience,
            SUM(tv_investment) AS total_tv_cost,
            SUM(insertions) AS total_insertions
        FROM tv
        GROUP BY campaign_id
    )

    -- Juntando tudo na base principal de campanhas
    SELECT
        c.campaign_name,
        c.category,

        -- Governança: Tratando nulos com COALESCE caso uma campanha não tenha rodado em um dos canais
        COALESCE(d.total_conversions, 0) AS conversions,
        (COALESCE(d.total_digital_cost, 0) + COALESCE(t.total_tv_cost, 0)) AS total_investment,

        -- Otimização de Conversão (CRO): Evitando divisão por zero com CASE WHEN
        CASE
            WHEN COALESCE(d.total_clicks, 0) > 0
            THEN ROUND((CAST(d.total_conversions AS FLOAT) / d.total_clicks) * 100, 2)
            ELSE 0
        END AS conversion_rate_pct,

        -- Métrica Estratégica (CAC): Custo total dividido pelas conversões
        CASE
            WHEN COALESCE(d.total_conversions, 0) > 0
            THEN ROUND((COALESCE(d.total_digital_cost, 0) + COALESCE(t.total_tv_cost, 0)) / CAST(d.total_conversions AS FLOAT), 2)
            ELSE NULL
        END AS cac_brl

    FROM campaigns c
    LEFT JOIN digital_metrics d ON c.campaign_id = d.campaign_id
    LEFT JOIN tv_metrics t ON c.campaign_id = t.campaign_id

    -- Foco em eficiência: ordenando pelos maiores custos de aquisição (CAC)
    ORDER BY cac_brl DESC;
"""

# Executa a query estruturada e exibe o resultado
df_resultado = pd.read_sql_query(query, conn)
display(df_resultado)

,campaign_name,category,conversions,total_investment,conversion_rate_pct,cac_brl
0,Campanha 18,Cassino,87583,2505498.44,3.25,28.61
1,Campanha 3,Esportes,117786,2977336.40,6.15,25.28
2,Campanha 9,Cassino,111174,2809936.84,8.36,25.28
3,Campanha 29,Streaming,113093,2628133.62,4.73,23.24
4,Campanha 30,Cassino,110586,2539801.93,5.83,22.97
5,Campanha 10,Streaming,114403,2626833.72,7.07,22.96
6,Campanha_8,Streaming,109595,2506909.79,5.53,22.87
7,Campanha 14,Esportes,116533,2647974.57,6.47,22.72
8,Campanha 1,Esportes,136121,3081774.20,5.99,22.64
9,Campanha 20,Cassino,114287,2572328.77,8.91,22.51


## 3. Output para Visualização
Com os dados validados e o CAC calculado e ordenado, a casa está arrumada.

O código abaixo apenas exporta o resultado final. Este ficheiro CSV será a fonte de dados única e tratada que vai alimentar o relatório visual que desenhei para a tomada de decisão da liderança.

In [3]:
# 1. Exporta o DataFrame para um ficheiro CSV sem a coluna de índice
df_resultado.to_csv('resultado_campanhas.csv', index=False)

# 2. Aciona a transferência (download) automática para o seu computador
from google.colab import files
files.download('resultado_campanhas.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>